# ETL — SIH + CNES
**Objetivo:** Carregar os dados brutos do SIH e CNES, filtrar internações por IAM (CID I21), realizar limpeza e integrar as duas bases.

**Outputs gerados:**
- `data/interim/sih_iam.parquet` — internações IAM limpas
- `data/interim/cnes_hospitais.parquet` — dados hospitalares consolidados
- `data/processed/base_modelagem.parquet` — base final (SIH + CNES)

## 0. Configurações e imports

In [8]:
import pandas as pd
import numpy as np
import os
import glob
from pathlib import Path

pd.set_option('display.max_columns', 60)
pd.set_option('display.float_format', '{:.2f}'.format)

# ── Caminhos ──────────────────────────────────────────────────────────────────
RAW_SIH     = Path('data/input/SIH')
RAW_CNES    = Path('data/input/CNES')
INTERIM     = Path('data/interim')
PROCESSED   = Path('data/processed')

INTERIM.mkdir(parents=True, exist_ok=True)
PROCESSED.mkdir(parents=True, exist_ok=True)

# ── Parâmetros ────────────────────────────────────────────────────────────────
# CIDs de interesse: I21 e seus subcódigos (I21.0 a I21.9)
CID_IAM = ['I21', 'I210', 'I211', 'I212', 'I213', 'I214',
            'I219', 'I21.0', 'I21.1', 'I21.2', 'I21.3',
            'I21.4', 'I21.9']

# Separador dos CSVs do pysus (ajuste se necessário)
SEP = ';'

# Encoding padrão DATASUS
ENCODING = 'latin-1'

print('✓ Configurações carregadas')
print(f'  SIH  → {RAW_SIH}')
print(f'  CNES → {RAW_CNES}')

✓ Configurações carregadas
  SIH  → data/input/SIH
  CNES → data/input/CNES


---
## 1. SIH — Internações por IAM
### 1.1 Carregamento e inspeção inicial

In [9]:
arquivos_sih = sorted(glob.glob(str(RAW_SIH / '*.csv')))
print(f'Arquivos SIH encontrados: {len(arquivos_sih)}')
for f in arquivos_sih:
    print(f'  {Path(f).name}')

Arquivos SIH encontrados: 12
  rdsp2501.csv
  rdsp2502.csv
  rdsp2503.csv
  rdsp2504.csv
  rdsp2505.csv
  rdsp2506.csv
  rdsp2507.csv
  rdsp2508.csv
  rdsp2509.csv
  rdsp2510.csv
  rdsp2511.csv
  rdsp2512.csv


In [10]:
# Inspeciona o primeiro arquivo para entender a estrutura
df_sample = pd.read_csv(arquivos_sih[0], sep=SEP, encoding=ENCODING, nrows=5)
print(f'Shape (amostra): {df_sample.shape}')
print(f'\nColunas ({len(df_sample.columns)}):')  
print(df_sample.columns.tolist())
df_sample.head(3)

Shape (amostra): (5, 1)

Colunas (1):
['UF_ZI,ANO_CMPT,MES_CMPT,ESPEC,CGC_HOSP,N_AIH,IDENT,CEP,MUNIC_RES,NASC,SEXO,UTI_MES_IN,UTI_MES_AN,UTI_MES_AL,UTI_MES_TO,MARCA_UTI,UTI_INT_IN,UTI_INT_AN,UTI_INT_AL,UTI_INT_TO,DIAR_ACOM,QT_DIARIAS,PROC_SOLIC,PROC_REA,VAL_SH,VAL_SP,VAL_SADT,VAL_RN,VAL_ACOMP,VAL_ORTP,VAL_SANGUE,VAL_SADTSR,VAL_TRANSP,VAL_OBSANG,VAL_PED1AC,VAL_TOT,VAL_UTI,US_TOT,DT_INTER,DT_SAIDA,DIAG_PRINC,DIAG_SECUN,COBRANCA,NATUREZA,NAT_JUR,GESTAO,RUBRICA,IND_VDRL,MUNIC_MOV,COD_IDADE,IDADE,DIAS_PERM,MORTE,NACIONAL,NUM_PROC,CAR_INT,TOT_PT_SP,CPF_AUT,HOMONIMO,NUM_FILHOS,INSTRU,CID_NOTIF,CONTRACEP1,CONTRACEP2,GESTRISCO,INSC_PN,SEQ_AIH5,CBOR,CNAER,VINCPREV,GESTOR_COD,GESTOR_TP,GESTOR_CPF,GESTOR_DT,CNES,CNPJ_MANT,INFEHOSP,CID_ASSO,CID_MORTE,COMPLEX,FINANC,FAEC_TP,REGCT,RACA_COR,ETNIA,SEQUENCIA,REMESSA,AUD_JUST,SIS_JUST,VAL_SH_FED,VAL_SP_FED,VAL_SH_GES,VAL_SP_GES,VAL_UCI,MARCA_UCI,DIAGSEC1,DIAGSEC2,DIAGSEC3,DIAGSEC4,DIAGSEC5,DIAGSEC6,DIAGSEC7,DIAGSEC8,DIAGSEC9,TPDISEC1,TPDISEC2,TPDISEC3,TP

,"UF_ZI,ANO_CMPT,MES_CMPT,ESPEC,CGC_HOSP,N_AIH,IDENT,CEP,MUNIC_RES,NASC,SEXO,UTI_MES_IN,UTI_MES_AN,UTI_MES_AL,UTI_MES_TO,MARCA_UTI,UTI_INT_IN,UTI_INT_AN,UTI_INT_AL,UTI_INT_TO,DIAR_ACOM,QT_DIARIAS,PROC_SOLIC,PROC_REA,VAL_SH,VAL_SP,VAL_SADT,VAL_RN,VAL_ACOMP,VAL_ORTP,VAL_SANGUE,VAL_SADTSR,VAL_TRANSP,VAL_OBSANG,VAL_PED1AC,VAL_TOT,VAL_UTI,US_TOT,DT_INTER,DT_SAIDA,DIAG_PRINC,DIAG_SECUN,COBRANCA,NATUREZA,NAT_JUR,GESTAO,RUBRICA,IND_VDRL,MUNIC_MOV,COD_IDADE,IDADE,DIAS_PERM,MORTE,NACIONAL,NUM_PROC,CAR_INT,TOT_PT_SP,CPF_AUT,HOMONIMO,NUM_FILHOS,INSTRU,CID_NOTIF,CONTRACEP1,CONTRACEP2,GESTRISCO,INSC_PN,SEQ_AIH5,CBOR,CNAER,VINCPREV,GESTOR_COD,GESTOR_TP,GESTOR_CPF,GESTOR_DT,CNES,CNPJ_MANT,INFEHOSP,CID_ASSO,CID_MORTE,COMPLEX,FINANC,FAEC_TP,REGCT,RACA_COR,ETNIA,SEQUENCIA,REMESSA,AUD_JUST,SIS_JUST,VAL_SH_FED,VAL_SP_FED,VAL_SH_GES,VAL_SP_GES,VAL_UCI,MARCA_UCI,DIAGSEC1,DIAGSEC2,DIAGSEC3,DIAGSEC4,DIAGSEC5,DIAGSEC6,DIAGSEC7,DIAGSEC8,DIAGSEC9,TPDISEC1,TPDISEC2,TPDISEC3,TPDISEC4,TPDISEC5,TPDISEC6,TPDISEC7,TPDISEC8,TPDISEC9"
0,"350000,2025,01,01,46374500028366,3525100117847..."
1,"350000,2025,01,02,46374500028366,3524130275908..."
2,"350000,2025,01,02,46374500028366,3524130278427..."


In [11]:
# Verifica qual coluna contém o CID principal
# No SIH/AIH a coluna costuma ser DIAG_PRINC
colunas_cid = [c for c in df_sample.columns if 'DIAG' in c.upper() or 'CID' in c.upper()]
print('Colunas com CID/DIAG:', colunas_cid)

# Verifica coluna de CNES do hospital
colunas_cnes = [c for c in df_sample.columns if 'CNES' in c.upper()]
print('Colunas CNES:', colunas_cnes)

Colunas com CID/DIAG: ['UF_ZI,ANO_CMPT,MES_CMPT,ESPEC,CGC_HOSP,N_AIH,IDENT,CEP,MUNIC_RES,NASC,SEXO,UTI_MES_IN,UTI_MES_AN,UTI_MES_AL,UTI_MES_TO,MARCA_UTI,UTI_INT_IN,UTI_INT_AN,UTI_INT_AL,UTI_INT_TO,DIAR_ACOM,QT_DIARIAS,PROC_SOLIC,PROC_REA,VAL_SH,VAL_SP,VAL_SADT,VAL_RN,VAL_ACOMP,VAL_ORTP,VAL_SANGUE,VAL_SADTSR,VAL_TRANSP,VAL_OBSANG,VAL_PED1AC,VAL_TOT,VAL_UTI,US_TOT,DT_INTER,DT_SAIDA,DIAG_PRINC,DIAG_SECUN,COBRANCA,NATUREZA,NAT_JUR,GESTAO,RUBRICA,IND_VDRL,MUNIC_MOV,COD_IDADE,IDADE,DIAS_PERM,MORTE,NACIONAL,NUM_PROC,CAR_INT,TOT_PT_SP,CPF_AUT,HOMONIMO,NUM_FILHOS,INSTRU,CID_NOTIF,CONTRACEP1,CONTRACEP2,GESTRISCO,INSC_PN,SEQ_AIH5,CBOR,CNAER,VINCPREV,GESTOR_COD,GESTOR_TP,GESTOR_CPF,GESTOR_DT,CNES,CNPJ_MANT,INFEHOSP,CID_ASSO,CID_MORTE,COMPLEX,FINANC,FAEC_TP,REGCT,RACA_COR,ETNIA,SEQUENCIA,REMESSA,AUD_JUST,SIS_JUST,VAL_SH_FED,VAL_SP_FED,VAL_SH_GES,VAL_SP_GES,VAL_UCI,MARCA_UCI,DIAGSEC1,DIAGSEC2,DIAGSEC3,DIAGSEC4,DIAGSEC5,DIAGSEC6,DIAGSEC7,DIAGSEC8,DIAGSEC9,TPDISEC1,TPDISEC2,TPDISEC3,TPDISEC4,TPDISEC5,

### 1.2 Carrega todos os meses e filtra IAM

In [ ]:
%%time

# Coluna do diagnóstico principal no SIH
COL_CID = 'DIAG_PRINC'  # ajuste se necessário após inspecionar acima

chunks = []
total_registros = 0

for arquivo in arquivos_sih:
    nome = Path(arquivo).name
    df = pd.read_csv(arquivo, sep=SEP, encoding=ENCODING, dtype=str, low_memory=False)
    total_registros += len(df)
    
    # Filtra I21 e subcódigos (com e sem ponto)
    mask = df[COL_CID].str.startswith('I21', na=False)
    df_iam = df[mask].copy()
    
    # Adiciona coluna de origem para rastreabilidade
    df_iam['_arquivo_origem'] = nome
    
    chunks.append(df_iam)
    print(f'  {nome}: {len(df):>8,} registros totais → {len(df_iam):>6,} IAM')

sih_iam = pd.concat(chunks, ignore_index=True)

print(f'\n─────────────────────────────────────')
print(f'Total registros SIH lidos : {total_registros:>10,}')
print(f'Internações IAM (I21)     : {len(sih_iam):>10,}')
print(f'Proporção IAM             : {len(sih_iam)/total_registros*100:.2f}%')

CPU times: user 591 ms, sys: 102 ms, total: 693 ms
Wall time: 693 ms


KeyError: 'DIAG_PRINC'

: 

### 1.3 Diagnóstico de qualidade dos dados SIH

In [ ]:
# ── Distribuição dos subcódigos I21 ───────────────────────────────────────────
print('Distribuição de subcódigos CID I21:')
print(sih_iam[COL_CID].value_counts().to_string())

In [ ]:
# ── Missing values ────────────────────────────────────────────────────────────
missing = sih_iam.isnull().sum()
missing_pct = (missing / len(sih_iam) * 100).round(2)
resumo_missing = pd.DataFrame({'missing': missing, 'pct': missing_pct})
resumo_missing = resumo_missing[resumo_missing['missing'] > 0].sort_values('pct', ascending=False)

print(f'Colunas com valores ausentes: {len(resumo_missing)}/{len(sih_iam.columns)}')
print(resumo_missing.to_string())

In [ ]:
# ── Variável-alvo: óbito hospitalar ───────────────────────────────────────────
# No SIH: MORTE == '1' indica óbito
# Verifica qual coluna representa o desfecho
colunas_obito = [c for c in sih_iam.columns 
                 if any(x in c.upper() for x in ['MORTE', 'OBITO', 'DESFECHO'])]
print('Colunas de desfecho encontradas:', colunas_obito)

if 'MORTE' in sih_iam.columns:
    print('\nDistribuição MORTE:')
    print(sih_iam['MORTE'].value_counts(dropna=False))
    taxa_obito = sih_iam['MORTE'].astype(str).eq('1').mean()
    print(f'\nTaxa de óbito hospitalar (IAM): {taxa_obito:.2%}')

### 1.4 Limpeza e tipagem do SIH

In [ ]:
# ── Colunas relevantes para o TCC ─────────────────────────────────────────────
# Selecione apenas as colunas que vai usar (reduz memória e deixa o dataset claro)
COLUNAS_SIH = [
    # Identificação
    'N_AIH',          # número da AIH (identificador único)
    'CNES',           # código CNES do hospital (chave para o join)
    
    # Desfecho (variável-alvo)
    'MORTE',          # 0=alta, 1=óbito
    
    # Tempo de internação
    'DT_INTER',       # data de internação
    'DT_SAIDA',       # data de saída
    'DIAS_PERM',      # dias de permanência
    
    # Diagnóstico
    'DIAG_PRINC',     # CID principal (I21.x)
    'DIAG_SECUN',     # CID secundário (comorbidades)
    
    # Paciente
    'IDADE',          # idade do paciente
    'SEXO',           # sexo
    'RACA_COR',       # raça/cor
    'MUNIC_RES',      # município de residência
    
    # Procedimento e financiamento
    'PROC_REA',       # procedimento realizado
    'COBRANCA',       # motivo da cobrança / tipo de saída
    'ESPEC',          # especialidade do leito
    'UTI_MES_TO',     # dias em UTI
    'VAL_TOT',        # valor total da internação
    
    # Geo
    'MUNIC_MOV',      # município do hospital
    
    # Rastreabilidade
    '_arquivo_origem',
]

# Filtra só colunas que existem no dataframe
colunas_disponiveis = [c for c in COLUNAS_SIH if c in sih_iam.columns]
colunas_ausentes = [c for c in COLUNAS_SIH if c not in sih_iam.columns]

if colunas_ausentes:
    print(f'⚠️  Colunas não encontradas (verifique nomes): {colunas_ausentes}')

sih_iam = sih_iam[colunas_disponiveis].copy()
print(f'\nDataFrame SIH IAM: {sih_iam.shape[0]:,} linhas × {sih_iam.shape[1]} colunas')

In [ ]:
# ── Tipagem ───────────────────────────────────────────────────────────────────

# Variável-alvo: garante binário 0/1
sih_iam['MORTE'] = pd.to_numeric(sih_iam['MORTE'], errors='coerce').fillna(0).astype(int)

# Datas
for col in ['DT_INTER', 'DT_SAIDA']:
    if col in sih_iam.columns:
        sih_iam[col] = pd.to_datetime(sih_iam[col], format='%Y%m%d', errors='coerce')

# Recalcula dias de permanência (mais confiável do que DIAS_PERM original)
if 'DT_INTER' in sih_iam.columns and 'DT_SAIDA' in sih_iam.columns:
    sih_iam['dias_internacao'] = (sih_iam['DT_SAIDA'] - sih_iam['DT_INTER']).dt.days
    # Sanity check: remove internações com dias negativos ou absurdos (> 365)
    n_invalidos = ((sih_iam['dias_internacao'] < 0) | (sih_iam['dias_internacao'] > 365)).sum()
    if n_invalidos > 0:
        print(f'⚠️  {n_invalidos} registros com dias de internação inválidos (< 0 ou > 365)')
    sih_iam = sih_iam[sih_iam['dias_internacao'].between(0, 365)].copy()

# Numéricos
for col in ['IDADE', 'DIAS_PERM', 'UTI_MES_TO', 'VAL_TOT']:
    if col in sih_iam.columns:
        sih_iam[col] = pd.to_numeric(sih_iam[col], errors='coerce')

# CNES como string com zeros à esquerda (7 dígitos)
sih_iam['CNES'] = sih_iam['CNES'].astype(str).str.zfill(7).str.strip()

# Extrai ano/mês da internação para análises temporais
if 'DT_INTER' in sih_iam.columns:
    sih_iam['ano_inter']  = sih_iam['DT_INTER'].dt.year
    sih_iam['mes_inter']  = sih_iam['DT_INTER'].dt.month

print('✓ Tipagem concluída')
print(sih_iam.dtypes)

In [ ]:
# ── Remove duplicatas ─────────────────────────────────────────────────────────
n_antes = len(sih_iam)
if 'N_AIH' in sih_iam.columns:
    sih_iam = sih_iam.drop_duplicates(subset='N_AIH', keep='first')
else:
    sih_iam = sih_iam.drop_duplicates()
n_depois = len(sih_iam)
print(f'Duplicatas removidas: {n_antes - n_depois:,}')
print(f'Registros únicos: {n_depois:,}')

In [ ]:
# ── Salva SIH interim ─────────────────────────────────────────────────────────
sih_iam.to_parquet(INTERIM / 'sih_iam.parquet', index=False)
print(f'✓ Salvo: {INTERIM}/sih_iam.parquet')
print(f'  Shape: {sih_iam.shape}')
print(f'  Taxa de óbito: {sih_iam["MORTE"].mean():.2%}')
sih_iam.describe(include='all').T.head(30)

---
## 2. CNES — Dados dos hospitais
### 2.1 Tabelas utilizadas

| Prefixo | Conteúdo | Uso |
|---------|----------|-----|
| `st` | Estabelecimentos | cadastro base, tipo, município |
| `lt` | Leitos | UTI, enfermaria, total |
| `eq` | Equipamentos | hemodinâmica, ecocardiograma |
| `sr` | Serviços especializados | cardiologia |
| `hb` | Habilitações | habilitação cardiologia/hemodinâmica |

In [ ]:
def carregar_cnes(prefixo: str, colunas: list = None) -> pd.DataFrame:
    """
    Carrega e concatena todos os arquivos CNES de um dado prefixo.
    Mantém a coluna CNES zerada com 7 dígitos para o join.
    """
    arquivos = sorted(glob.glob(str(RAW_CNES / f'{prefixo}sp25*.csv')))
    if not arquivos:
        print(f'  ⚠️  Nenhum arquivo encontrado para prefixo: {prefixo}')
        return pd.DataFrame()
    
    dfs = []
    for arq in arquivos:
        df = pd.read_csv(arq, sep=SEP, encoding=ENCODING, dtype=str, low_memory=False)
        df['_arquivo_cnes'] = Path(arq).name
        if colunas:
            cols = [c for c in colunas + ['CNES', '_arquivo_cnes'] if c in df.columns]
            df = df[cols]
        dfs.append(df)
    
    resultado = pd.concat(dfs, ignore_index=True)
    
    # Normaliza CNES
    if 'CNES' in resultado.columns:
        resultado['CNES'] = resultado['CNES'].astype(str).str.zfill(7).str.strip()
    
    # Remove duplicatas por CNES (mantém o mais recente = maior nome de arquivo)
    if 'CNES' in resultado.columns:
        resultado = resultado.sort_values('_arquivo_cnes').drop_duplicates(
            subset='CNES', keep='last'
        )
    
    print(f'  {prefixo}: {len(arquivos)} arquivos → {len(resultado):,} estabelecimentos únicos')
    return resultado

print('Função carregar_cnes() definida ✓')

### 2.2 Estabelecimentos (st) — cadastro base

In [ ]:
# Inspeciona colunas da tabela ST
arq_st = sorted(glob.glob(str(RAW_CNES / 'stsp25*.csv')))[0]
df_st_sample = pd.read_csv(arq_st, sep=SEP, encoding=ENCODING, nrows=3)
print('Colunas ST:', df_st_sample.columns.tolist())

In [ ]:
COLUNAS_ST = [
    'CNES',
    'RAZAO_SOCIAL', 'NOME_FANTASIA',
    'MUNIC_GESTR',   # município gestor
    'COD_CEP',
    'TP_GESTAO',     # tipo de gestão (E=estadual, M=municipal, D=dupla)
    'PF_PJ',         # pessoa física ou jurídica
    'CPF_CNPJ',
    'NATUREZA',      # natureza jurídica
    'ESFERA_ADM',    # esfera administrativa
    'RETENCAO',
    'TIPO_UNIDADE',  # tipo de unidade (hospital geral, especializado, etc.)
    'TP_PREST_SERVICO',  # prestador SUS
    'VINC_SUS',
    'NIVEL_DEP',
    'CNPJ_MANTENEDOR',
]

cnes_st = carregar_cnes('st', COLUNAS_ST)
print(f'Shape ST: {cnes_st.shape}')
cnes_st.head(3)

### 2.3 Leitos (lt)

In [ ]:
# Inspeciona colunas LT
arq_lt = sorted(glob.glob(str(RAW_CNES / 'ltsp25*.csv')))[0]
df_lt_sample = pd.read_csv(arq_lt, sep=SEP, encoding=ENCODING, nrows=3)
print('Colunas LT:', df_lt_sample.columns.tolist())

In [ ]:
# Carrega leitos e agrega por CNES
arquivos_lt = sorted(glob.glob(str(RAW_CNES / 'ltsp25*.csv')))
lt_raw = pd.concat([
    pd.read_csv(f, sep=SEP, encoding=ENCODING, dtype=str, low_memory=False) 
    for f in arquivos_lt
], ignore_index=True)

lt_raw['CNES'] = lt_raw['CNES'].astype(str).str.zfill(7)
print('Colunas disponíveis em LT:')
print(lt_raw.columns.tolist())
lt_raw.head(3)

In [ ]:
# ── Agregação de leitos por CNES ──────────────────────────────────────────────
# Colunas típicas: TP_LEITO (tipo), CODLEITO, QT_SUS, QT_EXIST
# Adapte conforme o output da célula anterior

for col in ['QT_SUS', 'QT_EXIST']:
    if col in lt_raw.columns:
        lt_raw[col] = pd.to_numeric(lt_raw[col], errors='coerce').fillna(0)

# Códigos de leito UTI no CNES:
# 74 = UTI adulto tipo I, 75 = tipo II, 76 = tipo III, 77 = coronariana
UTI_CODIGOS = ['74', '75', '76', '77']

if 'CODLEITO' in lt_raw.columns and 'QT_SUS' in lt_raw.columns:
    leitos_agg = lt_raw.groupby('CNES').agg(
        leitos_sus_total  = ('QT_SUS',   'sum'),
        leitos_exist_total= ('QT_EXIST', 'sum'),
    ).reset_index()
    
    leitos_uti = lt_raw[lt_raw['CODLEITO'].isin(UTI_CODIGOS)].groupby('CNES').agg(
        leitos_uti_sus   = ('QT_SUS',   'sum'),
        leitos_uti_exist = ('QT_EXIST', 'sum'),
    ).reset_index()
    
    cnes_leitos = leitos_agg.merge(leitos_uti, on='CNES', how='left').fillna(0)
    print(f'✓ Leitos agregados: {len(cnes_leitos):,} estabelecimentos')
    cnes_leitos.head(3)
else:
    print('⚠️  Colunas CODLEITO/QT_SUS não encontradas. Verifique os nomes no output anterior.')
    cnes_leitos = pd.DataFrame()

### 2.4 Equipamentos (eq) — hemodinâmica e diagnóstico cardíaco

In [ ]:
# Inspeciona EQ
arq_eq = sorted(glob.glob(str(RAW_CNES / 'eqsp25*.csv')))[0]
df_eq_sample = pd.read_csv(arq_eq, sep=SEP, encoding=ENCODING, nrows=3)
print('Colunas EQ:', df_eq_sample.columns.tolist())
df_eq_sample.head(3)

In [ ]:
# Equipamentos relevantes para IAM:
# 26 = Sala de hemodinâmica, 51 = Ecocardiograma, 64 = Tomógrafo
# (verifique os códigos no dicionário CNES — data/external/)

EQ_CARDIOLOGICOS = {
    '26': 'hemodinamica',
    '51': 'ecocardiograma',
    '64': 'tomografo',
}

arquivos_eq = sorted(glob.glob(str(RAW_CNES / 'eqsp25*.csv')))
eq_raw = pd.concat([
    pd.read_csv(f, sep=SEP, encoding=ENCODING, dtype=str, low_memory=False) 
    for f in arquivos_eq
], ignore_index=True)

eq_raw['CNES'] = eq_raw['CNES'].astype(str).str.zfill(7)

# Cria flags binárias por equipamento
if 'TIPEQUIP' in eq_raw.columns:
    cnes_equip = eq_raw.groupby('CNES').apply(
        lambda x: pd.Series({
            f'tem_{v}': int(k in x['TIPEQUIP'].values)
            for k, v in EQ_CARDIOLOGICOS.items()
        })
    ).reset_index()
    print(f'✓ Equipamentos: {len(cnes_equip):,} estabelecimentos')
    print(cnes_equip.describe().T)
else:
    print('⚠️  Coluna TIPEQUIP não encontrada. Verifique o nome no output anterior.')
    cnes_equip = pd.DataFrame()

### 2.5 Serviços (sr) — cardiologia

In [ ]:
arquivos_sr = sorted(glob.glob(str(RAW_CNES / 'srsp25*.csv')))
sr_raw = pd.concat([
    pd.read_csv(f, sep=SEP, encoding=ENCODING, dtype=str, low_memory=False) 
    for f in arquivos_sr
], ignore_index=True)

sr_raw['CNES'] = sr_raw['CNES'].astype(str).str.zfill(7)
print('Colunas SR:', sr_raw.columns.tolist())
sr_raw.head(3)

In [ ]:
# Serviço 131 = Cardiologia no CNES
# Serviço 134 = Hemodinâmica
SERVICOS_CARDIO = ['131', '134']

if 'SERV_ESP' in sr_raw.columns:
    cnes_servicos = sr_raw.groupby('CNES').apply(
        lambda x: pd.Series({
            'tem_servico_cardiologia'  : int('131' in x['SERV_ESP'].values),
            'tem_servico_hemodinamica' : int('134' in x['SERV_ESP'].values),
        })
    ).reset_index()
    print(f'✓ Serviços: {len(cnes_servicos):,} estabelecimentos')
    print(cnes_servicos[['tem_servico_cardiologia', 'tem_servico_hemodinamica']].sum())
else:
    print('⚠️  Coluna SERV_ESP não encontrada. Verifique o nome no output anterior.')
    cnes_servicos = pd.DataFrame()

### 2.6 Consolida base CNES

In [ ]:
# Parte do cadastro base (ST) e faz left joins
cnes_hospitais = cnes_st.copy()

tabelas_cnes = [
    (cnes_leitos,   'leitos'),
    (cnes_equip,    'equipamentos'),
    (cnes_servicos, 'serviços'),
]

for df_extra, nome in tabelas_cnes:
    if not df_extra.empty:
        n_antes = len(cnes_hospitais)
        cnes_hospitais = cnes_hospitais.merge(df_extra, on='CNES', how='left')
        print(f'  Merge {nome}: {n_antes} → {len(cnes_hospitais)} registros')

# Preenche zeros onde não há informação de leitos/equipamentos
cols_fill_zero = [c for c in cnes_hospitais.columns 
                  if c.startswith('leitos_') or c.startswith('tem_')]
cnes_hospitais[cols_fill_zero] = cnes_hospitais[cols_fill_zero].fillna(0)

# Salva
cnes_hospitais.to_parquet(INTERIM / 'cnes_hospitais.parquet', index=False)
print(f'\n✓ Salvo: {INTERIM}/cnes_hospitais.parquet')
print(f'  Shape: {cnes_hospitais.shape}')
cnes_hospitais.head(3)

---
## 3. Join SIH + CNES
### 3.1 Merge pela chave CNES

In [ ]:
# Recarrega os intermediários (garante reprodutibilidade desta seção isolada)
sih = pd.read_parquet(INTERIM / 'sih_iam.parquet')
cnes = pd.read_parquet(INTERIM / 'cnes_hospitais.parquet')

print(f'SIH IAM    : {len(sih):,} internações')
print(f'CNES       : {len(cnes):,} estabelecimentos')

# Hospitais únicos nas internações IAM
cnes_no_sih = sih['CNES'].nunique()
print(f'\nHospitais únicos no SIH IAM : {cnes_no_sih:,}')
print(f'Hospitais únicos no CNES    : {cnes["CNES"].nunique():,}')

# Cobertura do join
cnes_com_match = sih['CNES'].isin(cnes['CNES']).sum()
print(f'\nInternações com CNES encontrado: {cnes_com_match:,} ({cnes_com_match/len(sih):.1%})')

In [ ]:
# ── Left join: mantém todas as internações IAM ─────────────────────────────────
# Left join porque o hospital é o contexto, não o filtro
base = sih.merge(cnes, on='CNES', how='left', suffixes=('', '_cnes'))

n_sem_match = base['TIPO_UNIDADE'].isnull().sum() if 'TIPO_UNIDADE' in base.columns else 0
print(f'Shape após merge: {base.shape}')
print(f'Internações sem match CNES: {n_sem_match:,} ({n_sem_match/len(base):.1%})')

### 3.2 Validação do join

In [ ]:
print('=== Resumo da base final ===')
print(f'Total de internações IAM   : {len(base):,}')
print(f'Total de colunas           : {base.shape[1]}')
print(f'Taxa de óbito              : {base["MORTE"].mean():.2%}')
print(f'Hospitais únicos           : {base["CNES"].nunique():,}')

if 'ano_inter' in base.columns:
    print(f'\nPor ano:')
    print(base.groupby('ano_inter').agg(
        internacoes=('MORTE', 'count'),
        obitos=('MORTE', 'sum'),
        taxa_obito=('MORTE', 'mean')
    ).round(4).to_string())

In [ ]:
# ── Missing na base final ─────────────────────────────────────────────────────
missing_final = base.isnull().mean().sort_values(ascending=False)
missing_alto = missing_final[missing_final > 0.20]

print(f'Colunas com >20% missing ({len(missing_alto)}):')  
print(missing_alto.apply(lambda x: f'{x:.1%}').to_string())

### 3.3 Salva a base de modelagem

In [ ]:
# Remove colunas de rastreabilidade antes de salvar a base final
cols_meta = [c for c in base.columns if c.startswith('_')]
base_modelagem = base.drop(columns=cols_meta)

base_modelagem.to_parquet(PROCESSED / 'base_modelagem.parquet', index=False)

print(f'✓ Salvo: {PROCESSED}/base_modelagem.parquet')
print(f'  Shape   : {base_modelagem.shape}')
print(f'  Tamanho : {(PROCESSED / "base_modelagem.parquet").stat().st_size / 1e6:.1f} MB')
base_modelagem.dtypes

---
## 4. Checklist de qualidade

Antes de avançar para a EDA, confirme:

- [x] `sih_iam.parquet` gerado com N > 0 internações IAM
- [ ] Variável-alvo `MORTE` é binária (0/1), sem nulos
- [ ] Chave `CNES` está em formato 7 dígitos nas duas bases
- [ ] Cobertura do join > 90%
- [ ] Sem duplicatas por `N_AIH`
- [ ] `base_modelagem.parquet` salvo em `data/processed/`

In [ ]:
# ── Checklist automático ──────────────────────────────────────────────────────
checks = {
    'Base não vazia'              : len(base_modelagem) > 0,
    'MORTE é binária'             : set(base_modelagem['MORTE'].unique()).issubset({0, 1}),
    'MORTE sem nulos'             : base_modelagem['MORTE'].isnull().sum() == 0,
    'CNES com 7 dígitos'          : base_modelagem['CNES'].str.len().eq(7).all(),
    'Cobertura join > 90%'        : cnes_com_match / len(sih) > 0.90,
    'Arquivo parquet existe'      : (PROCESSED / 'base_modelagem.parquet').exists(),
}

for descricao, passou in checks.items():
    status = '✅' if passou else '❌'
    print(f'{status}  {descricao}')

if all(checks.values()):
    print('\n🎉 ETL concluído com sucesso! Próximo passo: 02_eda.ipynb')
else:
    print('\n⚠️  Revise os itens com ❌ antes de avançar.')